In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json, lit
from pyspark.sql.types import StructType
import glob
import os
import shutil

In [0]:
%run ../../../helper_function/helper_function


In [0]:
access_key = dbutils.secrets.get(scope="barcakings-secrets",key="s3-access-key")
secret_key = dbutils.secrets.get(scope="barcakings-secrets",key="s3-secret-key")
databricks_token = dbutils.secrets.get(scope="barcakings-secrets",key="dbk-pat")

In [0]:
dbutils.widgets.text("sport", "football", "sport")
sport = dbutils.widgets.get("sport")

In [0]:
# Read control table
control_df = spark.sql(
    f"""
        SELECT 
            *
        FROM  
            workspace.bk_consumable.consumable_control_tbl
        WHERE 
            sport = '{sport}' 
    """)
control_df.display()

In [0]:
control_rows = control_df.collect()

for row in control_rows:

    team = row["team"]
    tournament = row["tournament"]
    source_uc = row["source_uc"]
    source_uc_schema = row["source_uc_schema"]
    source_uc_table = row["source_uc_table"]
    target_uc = row["target_uc"]
    target_uc_schema = row["target_uc_schema"]
    target_uc_table = row["target_uc_table"]
    spark_view = row["spark_view"]
    sql_file_path = row["sql_file_path"]
    load_type = row["load_type"]
    delta_col = row["delta_col"]
    partition_col = row["partition_col"]
    target_s3_bucket = row["target_s3_bucket"]
    target_s3_key = row["target_s3_key"]

    print(target_uc_table)
    
    raw_df = spark.sql(
        f"SELECT * FROM {source_uc}.{source_uc_schema}.{source_uc_table}"
    )

    raw_df.createOrReplaceTempView(spark_view)

    try:
        with open(sql_file_path, "r") as file:
            sql_query = file.read()

    except Exception as e:
        raise Exception(
            f"Unable to read SQL file: {sql_file_path}. Error: {str(e)}"
        )
    
    df_consumable = spark.sql(sql_query)

    df_consumable = df_consumable.withColumn('rec_captured_date', F.current_date())

    df_consumable\
    .write.mode(load_type)\
    .partitionBy(partition_col)\
    .saveAsTable(f"{target_uc}.{target_uc_schema}.{target_uc_table}")

    base_path = "/Volumes/workspace/bk_consumable/barca_consumable_data"

    temp_path = f"{base_path}/{target_uc_table}_temp"
    final_file = f"{base_path}/{target_uc_table}.parquet"

    df_consumable.coalesce(1)\
    .write.format("parquet")\
    .mode("overwrite")\
    .option("header", "true")\
    .save(temp_path)


    # Find generated parquet
    files = dbutils.fs.ls(temp_path)

    part_file = [
        f.path for f in files
        if f.name.startswith("part-") and f.name.endswith(".parquet")
    ][0]

    # Move and rename
    dbutils.fs.mv(part_file, final_file)

    # Remove temporary directory
    dbutils.fs.rm(temp_path, True)

    with open(final_file, "rb" ) as file:
        parquet_file_data = file.read()

    #Uploading parquet to S3
    upload_s3_vol(
        file_data = parquet_file_data, 
        file_format = 'parquet',
        bucket_name = target_s3_bucket ,
        s3_key = f"{target_s3_key}{target_uc_table}.parquet",
        volume_path = None 
    )

    # Remove Volume Files
    dbutils.fs.rm(final_file, True)